In [ ]:
import json
import os
import re
import time
import collections
import ipaddress
import numpy as np
import pandas as pd
import requests
import joblib
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from IPython.display import display
from shared_variables import (
    IP_RANGES_DIR,
    PROVIDERS,
    HF_REPO_ID,
    HF_REPO_TYPE,
    CONSENSUS_DATA_FILENAME,
    EXECUTION_DATA_FILENAME,
)

In [ ]:
def read_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def read_jsonl_from_hf(filename):
    path = hf_hub_download(
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
        filename=filename,
    )
    return read_jsonl(path)

cl_records = read_jsonl_from_hf(CONSENSUS_DATA_FILENAME)
el_records = read_jsonl_from_hf(EXECUTION_DATA_FILENAME)

In [ ]:
cl_top = pd.DataFrame([{k: v for k, v in r.items() if k != "peer_properties"} for r in cl_records])
cl_props = pd.json_normalize([r["peer_properties"] for r in cl_records])
df_consensus_visits = pd.concat([cl_top, cl_props], axis=1)

el_top = pd.DataFrame([{k: v for k, v in r.items() if k != "peer_properties"} for r in el_records])
el_props = pd.json_normalize([r["peer_properties"] for r in el_records])
df_execution_visits = pd.concat([el_top, el_props], axis=1)

In [ ]:
df_el = df_execution_visits.copy()
df_cl = df_consensus_visits.copy()

In [ ]:
def extract_ip_from_connect_maddr(maddr):
    if not maddr:
        return pd.NA
    parts = str(maddr).split("/")
    if len(parts) >= 3 and parts[1] == "ip4":
        return parts[2]
    return pd.NA

df_el = df_el.copy()
df_el["ip"] = df_el["connect_maddr"].apply(extract_ip_from_connect_maddr)
df_el = df_el[df_el["ip"].notna()].copy()

df_cl = df_cl.copy()
df_cl["ip"] = df_cl["connect_maddr"].apply(extract_ip_from_connect_maddr)
df_cl = df_cl[df_cl["ip"].notna()].copy()

print(f"EL rows with IP: {len(df_el)}")
print(f"CL rows with IP: {len(df_cl)}")

df_cl_pair = df_cl.copy()
df_el_pair = df_el.copy()

df_cl_pair["ip_occurrence"] = df_cl_pair.groupby("ip").cumcount()
df_el_pair["ip_occurrence"] = df_el_pair.groupby("ip").cumcount()

df_cl_renamed = df_cl_pair.rename(columns=lambda c: c + "_cl" if c not in {"ip", "ip_occurrence"} else c)
df_el_renamed = df_el_pair.rename(columns=lambda c: c + "_el" if c not in {"ip", "ip_occurrence"} else c)

df_merged = df_cl_renamed.merge(df_el_renamed, on=["ip", "ip_occurrence"], how="inner")

print(df_merged.shape)

In [ ]:
df_merged = df_merged[
    (df_merged["fork_digest_cl"].notna())
].copy()

print(df_merged.shape)

## Preprocessing and Feature Assignment

In [ ]:
CL_CLIENT_NAMES = ["lighthouse", "prysm", "teku", "nimbus", "lodestar", "grandine", "caplin"]
EL_CLIENT_NAMES = ["geth", "nethermind", "besu", "erigon", "reth", "nimbus", "ethrex"]

In [ ]:
_CL_PATTERN_SOURCES = {
    "lighthouse": re.compile(r"^lighthouse", re.IGNORECASE),
    "prysm":      re.compile(r"^prysm",      re.IGNORECASE),
    "teku":       re.compile(r"^teku",        re.IGNORECASE),
    "nimbus":     re.compile(r"^nimbus",      re.IGNORECASE),
    "lodestar":   re.compile(r"^lodestar",    re.IGNORECASE),
    "grandine":   re.compile(r"^grandine",    re.IGNORECASE),
    "caplin":     re.compile(r"caplin",      re.IGNORECASE),
}

_EL_PATTERN_SOURCES = {
    "geth":       re.compile(r"^geth",      re.IGNORECASE),
    "nethermind": re.compile(r"^nethermind", re.IGNORECASE),
    "besu":       re.compile(r"^besu",      re.IGNORECASE),
    "erigon":     re.compile(r"^erigon",     re.IGNORECASE),
    "reth":       re.compile(r"^reth",      re.IGNORECASE),
    "nimbus":     re.compile(r"^nimbus",      re.IGNORECASE),
    "ethrex":     re.compile(r"^ethrex",      re.IGNORECASE),
}

_CL_PATTERNS = [(name, _CL_PATTERN_SOURCES[name]) for name in CL_CLIENT_NAMES]
_EL_PATTERNS = [(name, _EL_PATTERN_SOURCES[name]) for name in EL_CLIENT_NAMES]

def parse_client_name(agent, patterns):
    return next((name for name, pat in patterns if pat.search(agent)), pd.NA)

df_merged["consensus_client"] = df_merged["agent_version_cl"].apply(lambda v: parse_client_name(v, _CL_PATTERNS))
df_merged["execution_client"] = df_merged["agent_version_el"].apply(lambda v: parse_client_name(v, _EL_PATTERNS))

In [ ]:
def _syncnets_count(s):
    try:
        return bin(int(s, 16)).count("1")
    except Exception:
        return 0

df_merged["syncnets_num"] = df_merged["syncnets_cl"].apply(lambda s: _syncnets_count(str(s)))

print(df_merged.shape)

In [ ]:
_ARM_TOKENS = {"aarch64", "aarch_64", "arm64"}
_X86_TOKENS = {"x86_64", "amd64", "linux-x64", "windows-x64", "linux-386", "x86_64-unknown-linux-gnu"}


def parse_hw_arch(agent):
    low = agent.lower()
    is_arm = any(tok in low for tok in _ARM_TOKENS)
    is_x86 = any(tok in low for tok in _X86_TOKENS)
    if is_arm and is_x86:
        return pd.NA
    if is_arm:
        return "ARM"
    if is_x86:
        return "x86"
    return pd.NA

def resolve_arch_and_os(c, e):
    if pd.isna(c):
        return e
    if pd.isna(e):
        return c
    if c == e:
        return c
    return pd.NA

cl_arches = df_merged["agent_version_cl"].apply(parse_hw_arch)
el_arches = df_merged["agent_version_el"].apply(parse_hw_arch)
df_merged["hw_arch"] = [resolve_arch_and_os(c, e) for c, e in zip(cl_arches, el_arches)]

print(df_merged.shape)

In [ ]:
_OS_TOKEN_GROUPS = {
    "linux":   {"linux"},
    "macos":   {"macos", "darwin", "osx"},
    "windows": {"windows"},
}

def parse_os_token(agent):
    low = agent.lower()
    return next((os_name for os_name, toks in _OS_TOKEN_GROUPS.items() if any(tok in low for tok in toks)), pd.NA)

cl_os_tokens = df_merged["agent_version_cl"].apply(parse_os_token)
el_os_tokens = df_merged["agent_version_el"].apply(parse_os_token)
df_merged["os_token"] = [resolve_arch_and_os(c, e) for c, e in zip(cl_os_tokens, el_os_tokens)]

print(df_merged.shape)

In [ ]:
def load_ip_indices():
    indices = []
    for provider in PROVIDERS:
        path = IP_RANGES_DIR / f"{provider}.json"
        data = json.loads(path.read_text())
        nets = [ipaddress.ip_network(p, strict=False) for p in data["prefixes"]]
        idx = collections.defaultdict(list)
        for net in nets:
            start = int(net.network_address) >> 24
            end = int(net.broadcast_address) >> 24
            for bucket in range(start, end + 1):
                idx[bucket].append(net)
        indices.append((provider, dict(idx)))
    return indices

def classify_ip(ip_str, indices):
    addr = ipaddress.ip_address(ip_str)
    for provider, idx in indices:
        if any(addr in net for net in idx.get(int(addr) >> 24, [])):
            return provider
    return pd.NA

indices = load_ip_indices()
df_merged["cloud_provider"] = df_merged["ip"].apply(lambda ip: classify_ip(ip, indices))

In [ ]:
df_merged["attnets_num_cl"] = pd.to_numeric(df_merged["attnets_num_cl"], errors="coerce").fillna(0).astype(int)

## Node Power

In [ ]:
client_fields_present = (
    df_merged["consensus_client"].notna()
    & df_merged["execution_client"].notna()
    & df_merged["hw_arch"].notna()
    & df_merged["os_token"].notna()
)
caplin_invalid_mask = (df_merged["consensus_client"] == "caplin") & (df_merged["execution_client"] != "erigon")

df = df_merged[client_fields_present & ~caplin_invalid_mask].copy()

In [ ]:
_KEEP_FOR_INFERENCE = {
    "consensus_client": "consensus_client",
    "attnets_num_cl": "attnets_num",
    "syncnets_num": "syncnets_num",
    "execution_client": "execution_client",
    "cloud_provider": "cloud_provider",
    "os_token": "os_token",
    "hw_arch": "hw_arch",
}

df = df[[c for c in _KEEP_FOR_INFERENCE]]
df = df.rename(columns=_KEEP_FOR_INFERENCE)

print(df.shape)

In [ ]:
_ATTNETS_SATURATION_THRESHOLD = 64
_GOSSIP_PHASE_NO_VALIDATORS = "no_validators"
_GOSSIP_PHASE_RAMPING = "ramping"
_GOSSIP_PHASE_SATURATED = "saturated"

def _gossip_phase(attnets_num):
    n = int(attnets_num)
    if n == 0:
        return _GOSSIP_PHASE_NO_VALIDATORS
    if n < _ATTNETS_SATURATION_THRESHOLD:
        return _GOSSIP_PHASE_RAMPING
    return _GOSSIP_PHASE_SATURATED

df["gossip_phase"] = df["attnets_num"].apply(_gossip_phase)
df["is_sync_committee_member"] = df["syncnets_num"] > 0
df["is_attnets_active"] = df["gossip_phase"] != _GOSSIP_PHASE_NO_VALIDATORS
df["is_validator"] = df["is_attnets_active"] | df["is_sync_committee_member"]

print(df.shape)

### Bare-Metal

In [ ]:
_TIER_WEIGHTS = {5: 0.75, 6: 0.25}
_IDLE = {5: 25.04, 6: 78.17}

_CL_MARGINAL = {
    "lighthouse": {5: 3.14,  6: 18.84},
    "prysm":      {5: 2.87,  6: 24.33},
    "teku":       {5: 3.32,  6: 27.46},
    "nimbus":     {5: 2.08,  6: 17.11},
    "lodestar":   {5: 3.89,  6: 33.55},
}

_EL_MARGINAL = {
    "geth":   {5: 9.70,  6: 47.70},
    "erigon": {5: 17.59, 6: 44.62},
    "besu":   {5: 31.02, 6: 75.04},
}

def _weighted(per_tier):
    return sum(_TIER_WEIGHTS[t] * v for t, v in per_tier.items())

WEIGHTED_IDLE_W = _weighted(_IDLE)
COMBINED_ADJUSTMENT_FACTOR = 0.91

CCRI_CL_MARGINAL_W = {k: _weighted(v) for k, v in _CL_MARGINAL.items()}
CCRI_EL_MARGINAL_W = {k: _weighted(v) for k, v in _EL_MARGINAL.items()}

PROXY_CL_MARGINAL_W = {
    "grandine": CCRI_CL_MARGINAL_W["nimbus"],
    "caplin":   CCRI_CL_MARGINAL_W["nimbus"],
}

PROXY_EL_MARGINAL_W = {
    "nethermind": CCRI_EL_MARGINAL_W["geth"],
    "reth":       CCRI_EL_MARGINAL_W["erigon"],
    "nimbus":     CCRI_EL_MARGINAL_W["geth"],
    "ethrex":     CCRI_EL_MARGINAL_W["erigon"],
}

_ALL_CL_MARGINAL_W = {**CCRI_CL_MARGINAL_W, **PROXY_CL_MARGINAL_W}
_ALL_EL_MARGINAL_W = {**CCRI_EL_MARGINAL_W, **PROXY_EL_MARGINAL_W}

if set(_ALL_CL_MARGINAL_W) != set(CL_CLIENT_NAMES):
    raise ValueError(f"CL marginal power coverage mismatch: {set(CL_CLIENT_NAMES) ^ set(_ALL_CL_MARGINAL_W)}")

if set(_ALL_EL_MARGINAL_W) != set(EL_CLIENT_NAMES):
    raise ValueError(f"EL marginal power coverage mismatch: {set(EL_CLIENT_NAMES) ^ set(_ALL_EL_MARGINAL_W)}")

ARM_LINUX_NODE_W_WEB3PI = 10.0

_MACOS_IDLE_SAMPLES_W = [6.8, 7.0]
ARM_MACOS_IDLE_W = sum(_MACOS_IDLE_SAMPLES_W) / len(_MACOS_IDLE_SAMPLES_W)
x86_MACOS_IDLE_W = 19.9

In [ ]:
df["power_cl_marginal_w"] = df["consensus_client"].map(_ALL_CL_MARGINAL_W)
df["power_el_marginal_w"] = df["execution_client"].map(_ALL_EL_MARGINAL_W)

arm_non_cloud = (df["hw_arch"] == "ARM") & df["cloud_provider"].isna()
x86_non_cloud = (df["hw_arch"] == "x86") & df["cloud_provider"].isna()

os_linux = df["os_token"].isin(_OS_TOKEN_GROUPS["linux"])
os_windows = df["os_token"].isin(_OS_TOKEN_GROUPS["windows"])
os_macos = df["os_token"].isin(_OS_TOKEN_GROUPS["macos"])

arm_linux_mask = arm_non_cloud & os_linux
arm_macos_mask = arm_non_cloud & os_macos
arm_windows_mask = arm_non_cloud & os_windows

x86_linux_mask = x86_non_cloud & os_linux 
x86_macos_mask = x86_non_cloud & os_macos
x86_windows_mask = x86_non_cloud & os_windows

df.loc[arm_linux_mask, "power_node_w"] = ARM_LINUX_NODE_W_WEB3PI

df.loc[x86_linux_mask | x86_windows_mask, "power_node_w"] = (
    (df.loc[x86_linux_mask | x86_windows_mask, "power_cl_marginal_w"] + df.loc[x86_linux_mask | x86_windows_mask, "power_el_marginal_w"])
    * COMBINED_ADJUSTMENT_FACTOR
    + WEIGHTED_IDLE_W
)

df.loc[arm_macos_mask, "power_node_w"] = (
    (df.loc[arm_macos_mask, "power_cl_marginal_w"] + df.loc[arm_macos_mask, "power_el_marginal_w"])
    * COMBINED_ADJUSTMENT_FACTOR
    + ARM_MACOS_IDLE_W
)

df.loc[x86_macos_mask, "power_node_w"] = (
    (df.loc[x86_macos_mask, "power_cl_marginal_w"] + df.loc[x86_macos_mask, "power_el_marginal_w"])
    * COMBINED_ADJUSTMENT_FACTOR
    + x86_MACOS_IDLE_W
)

print(df.shape)

In [ ]:
df = df.loc[~arm_windows_mask].copy()

print(df.shape)

### Cloud

In [ ]:
SSD_OVERHEAD_W_PANKOVSKA = 5.0
CLOUD_PUE_PANKOVSKA = 1.2

CLOUD_PUE_CCF = 1.185

CCF_MEMORY_W_PER_GB = 0.392
CCF_SSD_W_PER_TB = 1.2
NODE_SSD_NON_VALIDATOR_TB = 2.0
NODE_SSD_VALIDATOR_TB = 4.0

NODE_VCPU_MIN_NON_VALIDATOR = 4
NODE_VCPU_MIN_VALIDATOR = 8
NODE_RAM_NON_VALIDATOR_GB = 32
NODE_RAM_VALIDATOR_GB = 64

_M6I_INSTANCES = {
    "m6i.2xlarge": {"vcpu": 8,  "ram_gb": 32,  "arch": "x86", "cpu": "Xeon Platinum 8375C", "pkg_w_100": 38.20, "ram_w_100": 19.20, "delta": 7.5,  "pct100": 64.90},
    "m6i.4xlarge": {"vcpu": 16, "ram_gb": 64,  "arch": "x86", "cpu": "Xeon Platinum 8375C", "pkg_w_100": 76.39, "ram_w_100": 38.40, "delta": 15.0, "pct100": 129.79},
}

_R6I_INSTANCES = {
    "r6i.2xlarge": {"vcpu": 8,  "ram_gb": 64,  "arch": "x86", "cpu": "Xeon Platinum 8375C", "pkg_w_100": _M6I_INSTANCES["m6i.2xlarge"]["pkg_w_100"], "ram_w_100": 2 * _M6I_INSTANCES["m6i.2xlarge"]["ram_w_100"], "delta": _M6I_INSTANCES["m6i.2xlarge"]["delta"], "pct100": _M6I_INSTANCES["m6i.2xlarge"]["pkg_w_100"] + 2 * _M6I_INSTANCES["m6i.2xlarge"]["ram_w_100"] + _M6I_INSTANCES["m6i.2xlarge"]["delta"]},
    "r6i.4xlarge": {"vcpu": 16, "ram_gb": 128, "arch": "x86", "cpu": "Xeon Platinum 8375C", "pkg_w_100": _M6I_INSTANCES["m6i.4xlarge"]["pkg_w_100"], "ram_w_100": 2 * _M6I_INSTANCES["m6i.4xlarge"]["ram_w_100"], "delta": _M6I_INSTANCES["m6i.4xlarge"]["delta"], "pct100": _M6I_INSTANCES["m6i.4xlarge"]["pkg_w_100"] + 2 * _M6I_INSTANCES["m6i.4xlarge"]["ram_w_100"] + _M6I_INSTANCES["m6i.4xlarge"]["delta"]},
}

_R6G_INSTANCES = {
    "r6g.2xlarge": {"vcpu": 8,  "ram_gb": 64,  "arch": "ARM", "pkg_w_100": 19.10, "ram_w_100": 38.40, "delta": 3.75, "pct100": 61.20},
    "r6g.4xlarge": {"vcpu": 16, "ram_gb": 128, "arch": "ARM", "pkg_w_100": 38.20, "ram_w_100": 76.80, "delta": 7.50, "pct100": 122.50},
}

_GRAVITON3_CHIP_W_100 = 100.0
_GRAVITON3_TOTAL_VCPUS = 64

_M6G_INSTANCES = {
    "m6g.2xlarge": {"vcpu": 8,  "ram_gb": 32, "arch": "ARM", "pkg_w_100": 19.10, "ram_w_100": 19.20, "delta": 3.8},
    "m6g.4xlarge": {"vcpu": 16, "ram_gb": 64, "arch": "ARM", "pkg_w_100": 38.20, "ram_w_100": 38.40, "delta": 7.5},
}

_M7G_INSTANCES_BASE = {
    "m7g.2xlarge": _M6G_INSTANCES["m6g.2xlarge"],
    "m7g.4xlarge": _M6G_INSTANCES["m6g.4xlarge"],
}

_M7G_INSTANCES_PKG = {
    name: {
        **v,
        "pkg_w_100": round(_GRAVITON3_CHIP_W_100 * (v["vcpu"] / _GRAVITON3_TOTAL_VCPUS), 2),
    }
    for name, v in _M7G_INSTANCES_BASE.items()
}

_M7G_INSTANCES = {
    name: {
        "vcpu":      v["vcpu"],
        "ram_gb":    v["ram_gb"],
        "arch":      v["arch"],
        "pkg_w_100": v["pkg_w_100"],
        "ram_w_100": v["ram_w_100"],
        "delta":     v["delta"],
        "pct100":    v["pkg_w_100"] + v["ram_w_100"] + v["delta"],
    }
    for name, v in _M7G_INSTANCES_PKG.items()
}

AWS_EC2_INSTANCE_POWER_W = {
    name: {"vcpu": v["vcpu"], "ram_gb": v["ram_gb"], "arch": v["arch"], "pct100": v["pct100"]}
    for name, v in {**_M6I_INSTANCES, **_R6I_INSTANCES, **_R6G_INSTANCES, **_M7G_INSTANCES}.items()
}

CCF_VCPU_MAX_W = {
    "aws":   3.50,
    "gcp":   4.26,
    "azure": 3.76,
}

_CCF_MICROARCH_MAX_W = {
    "EPYC 1st Gen":    2.6042,
    "EPYC 2nd Gen":    1.6930,
    "EPYC 3rd Gen":    1.9573,
    "EPYC 4th Gen":    2.2822,
    "EPYC 5th Gen":    8.9614,
    "Cascade Lake":    4.0632,
    "Ice Lake":        3.7582,
    "Sapphire Rapids": 4.1605,
    "Skylake":         4.1042,
}

_PROVIDER_MICROARCHS = {
    "hetzner":      ["EPYC 2nd Gen", "EPYC 3rd Gen", "EPYC 4th Gen"],
    "ovh":          ["EPYC 3rd Gen", "EPYC 4th Gen", "Cascade Lake", "Sapphire Rapids"],
    "contabo":      ["EPYC 2nd Gen", "EPYC 4th Gen", "EPYC 5th Gen"],
    "netcup":       ["EPYC 2nd Gen", "EPYC 4th Gen", "EPYC 5th Gen"],
    "digitalocean": ["Skylake", "Cascade Lake", "Ice Lake", "Sapphire Rapids", "EPYC 2nd Gen", "EPYC 3rd Gen"],
    "vultr":        ["Cascade Lake", "EPYC 2nd Gen", "EPYC 3rd Gen"],
    "linode":       ["EPYC 1st Gen", "EPYC 2nd Gen", "EPYC 3rd Gen"],
    "leaseweb":     ["Cascade Lake", "Sapphire Rapids", "EPYC 2nd Gen", "EPYC 3rd Gen"],
    "clouvider":    ["Cascade Lake", "EPYC 3rd Gen"],
    "latitude":     ["EPYC 3rd Gen", "EPYC 4th Gen", "EPYC 5th Gen"],
    "oracle":       ["EPYC 3rd Gen", "EPYC 4th Gen", "EPYC 5th Gen"],
}

NON_HYPERSCALE_VCPU_MAX_W = {
    provider: round(
        sum(_CCF_MICROARCH_MAX_W[arch] for arch in archs) / len(archs),
        4,
    )
    for provider, archs in _PROVIDER_MICROARCHS.items()
}

_ALL_VCPU_MAX_W = {**CCF_VCPU_MAX_W, **NON_HYPERSCALE_VCPU_MAX_W}

if set(_ALL_VCPU_MAX_W) != set(PROVIDERS):
    raise ValueError(f"vCPU max power coverage mismatch: {set(PROVIDERS) ^ set(_ALL_VCPU_MAX_W)}")

In [ ]:
df["required_ram_gb"] = np.where(df["is_validator"], NODE_RAM_VALIDATOR_GB, NODE_RAM_NON_VALIDATOR_GB)
df["required_vcpu"] = np.where(df["is_validator"], NODE_VCPU_MIN_VALIDATOR, NODE_VCPU_MIN_NON_VALIDATOR)
df["required_ssd_tb"] = np.where(df["is_validator"], NODE_SSD_VALIDATOR_TB, NODE_SSD_NON_VALIDATOR_TB)

aws_mask = df["cloud_provider"] == "aws"
for idx, row in df[aws_mask].iterrows():
    node_arch = row["hw_arch"]
    candidates = {
        k: v for k, v in AWS_EC2_INSTANCE_POWER_W.items()
        if v["vcpu"] >= row["required_vcpu"]
        and v["ram_gb"] >= row["required_ram_gb"]
        and v["arch"] == node_arch
    }

    if not candidates:
        raise ValueError(f"No EC2 candidates found.")

    pct100_vals = [v["pct100"] for v in candidates.values()]
    p_cloud_load = sum(pct100_vals) / len(pct100_vals)
    df.at[idx, "power_node_w"] = (p_cloud_load + SSD_OVERHEAD_W_PANKOVSKA) * CLOUD_PUE_PANKOVSKA

non_aws_cloud_mask = df["cloud_provider"].notna() & (df["cloud_provider"] != "aws")
for idx, row in df[non_aws_cloud_mask].iterrows():
    vcpu_min = row["required_vcpu"]
    p_cpu_w = _ALL_VCPU_MAX_W[row["cloud_provider"]] * vcpu_min
    p_ram_w = CCF_MEMORY_W_PER_GB * row["required_ram_gb"]
    p_ssd_w = CCF_SSD_W_PER_TB * row["required_ssd_tb"]
    df.at[idx, "power_node_w"] = (p_cpu_w + p_ram_w + p_ssd_w) * CLOUD_PUE_CCF

print(df.shape)

## Prediction Model

In [ ]:
train_data = df.copy()

In [ ]:
_KEEP_FOR_TRAIN_DATA = [
    "consensus_client",
    "execution_client",
    "hw_arch",
    "os_token",
    "cloud_provider",
    "attnets_num",
    "syncnets_num",
    "power_node_w",
]

train_data = train_data[_KEEP_FOR_TRAIN_DATA]

print(train_data.shape)

In [ ]:
CATEGORICAL_FEATURES = [
    "consensus_client",
    "execution_client",
    "hw_arch",
    "os_token",
    "cloud_provider",
]
NUMERICAL_FEATURES = ["attnets_num", "syncnets_num"]
TARGET = "power_node_w"

MODEL_PATH = "rf_power_model.joblib"

N_ESTIMATORS = 500
TEST_SIZE = 0.2
RANDOM_STATE = 42


def build_pipeline():
    preprocessor = ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES)],
        remainder="passthrough",
    )
    model = RandomForestRegressor(
        n_estimators=N_ESTIMATORS,
        oob_score=True,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    return Pipeline([("preprocess", preprocessor), ("model", model)])


def grouped_importances(pipeline):
    feature_names = pipeline.named_steps["preprocess"].get_feature_names_out()
    importances = pipeline.named_steps["model"].feature_importances_
    grouped = {}
    for name, importance in zip(feature_names, importances):
        matched = next(
            (feat for feat in CATEGORICAL_FEATURES if name.split("__")[-1].startswith(f"{feat}_")),
            None,
        )
        key = matched or name.split("__")[-1]
        grouped[key] = grouped.get(key, 0.0) + float(importance)
    return dict(sorted(grouped.items(), key=lambda item: -item[1]))


features = df[CATEGORICAL_FEATURES + NUMERICAL_FEATURES]
target = df[TARGET]

x_train, x_test, y_train, y_test = train_test_split(
    features, target, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

pipeline = build_pipeline()
pipeline.fit(x_train, y_train)

predictions = pipeline.predict(x_test)
metrics = {
    "n_train": len(x_train),
    "n_test": len(x_test),
    "rmse": mean_squared_error(y_test, predictions) ** 0.5,
    "mae": mean_absolute_error(y_test, predictions),
    "r2": r2_score(y_test, predictions),
    "oob_r2": pipeline.named_steps["model"].oob_score_,
    "y_test_mean": float(y_test.mean()),
    "y_test_std": float(y_test.std()),
    "feature_importances": grouped_importances(pipeline),
}

print(json.dumps(metrics, indent=2))

joblib.dump(pipeline, MODEL_PATH)

In [ ]:
def split_recoverable_and_invalid(df_unfiltered):
    missing_required = (
        df_unfiltered["hw_arch"].isna()
        | df_unfiltered["consensus_client"].isna()
        | df_unfiltered["execution_client"].isna()
        | df_unfiltered["os_token"].isna()
    )
    caplin_invalid_mask = (df_unfiltered["consensus_client"] == "caplin") & (df_unfiltered["execution_client"] != "erigon")

    recoverable_df = df_unfiltered[missing_required & ~caplin_invalid_mask].copy()
    invalid_df = df_unfiltered[caplin_invalid_mask & ~missing_required].copy()
    return recoverable_df, invalid_df


df_unfiltered = df_merged[[c for c in _KEEP_FOR_INFERENCE]].rename(columns=_KEEP_FOR_INFERENCE).copy()

recoverable, invalid = split_recoverable_and_invalid(df_unfiltered)

recoverable[CATEGORICAL_FEATURES] = recoverable[CATEGORICAL_FEATURES].fillna("unknown")

recoverable["power_node_w"] = pipeline.predict(recoverable[CATEGORICAL_FEATURES + NUMERICAL_FEATURES])
recoverable["power_source"] = "model_estimate"

labelled = df.copy()
labelled["power_source"] = "rule_based"

output_columns = CATEGORICAL_FEATURES + NUMERICAL_FEATURES + ["power_node_w", "power_source"]
extended = pd.concat(
    [labelled[output_columns], recoverable[output_columns]],
    ignore_index=True,
)

print(f"rule_based rows        : {len(labelled)}")
print(f"model_estimate rows     : {len(recoverable)}")
print(f"unrecoverable rows      : {len(invalid)}")
print(f"total matched peers     : {len(df_unfiltered)}")
print(f"coverage after extension: {len(extended)} / {len(df_unfiltered)}")

In [ ]:
_NIMBUS_GETH_CSV      = "reports_nimbus_geth.csv"
_PRYSM_NETHERMIND_CSV = None
_IDLE_CSV             = None

_RAW_POWER_COL = "power (Watt)"
_PLOT_DATE     = "2026-07-11"

def _load_power_csv(filename):
    path = hf_hub_download(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, filename=filename)
    df = pd.read_csv(path, parse_dates=["time"])
    df = df.rename(columns={_RAW_POWER_COL: "power"})[["time", "power"]]
    return df.sort_values("time").reset_index(drop=True)

def _filter_day(df, date_str):
    return df[df["time"].dt.date.astype(str) == date_str].copy()

df_ng_full = _load_power_csv(_NIMBUS_GETH_CSV)
df_ng_day  = _filter_day(df_ng_full, _PLOT_DATE)

print(f"Nimbus+Geth  total rows       : {len(df_ng_full)}")
print(f"Nimbus+Geth  {_PLOT_DATE} rows : {len(df_ng_day)}")

In [ ]:
_SMOOTH_WINDOW = 20

fig, ax = plt.subplots(figsize=(13, 4))

hours  = (df_ng_day["time"] - df_ng_day["time"].dt.normalize()).dt.total_seconds() / 3600
smooth = df_ng_day["power"].rolling(_SMOOTH_WINDOW, center=True, min_periods=1).mean()

ax.plot(hours, df_ng_day["power"], color="#d97706", alpha=0.18, linewidth=0.5)
ax.plot(hours, smooth, color="#d97706", linewidth=1.8, label="Nimbus (CL) + Geth (EL)")

ax.set_xlim(0, 24)
ax.set_xticks(range(0, 25, 2))
ax.set_xticklabels([f"{h:02d}:00" for h in range(0, 25, 2)], rotation=30, ha="right")
ax.set_xlabel(f"Time of day (Saturday {_PLOT_DATE})")
ax.set_ylabel("Power (W)")
ax.legend(loc="upper right", framealpha=0.9, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(f"Node Power Consumption — Nimbus + Geth — Saturday {_PLOT_DATE}", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"power_ng_{_PLOT_DATE}.png", dpi=150, bbox_inches="tight")
plt.show()

## Monitoring

In [ ]:
#display(df_merged[["hw_arch", "os_token", "agent_version_cl", "agent_version_el", "consensus_client", "execution_client"]].head(300))

# display(df_merged["agent_version_cl"].describe())

for col in df_merged.columns:
    print(f"Column: {col}")
    display(df_merged[col].value_counts(dropna=False).to_frame("count"))

# display(df["power_node_w"].describe())
# display(df_merged["syncnets_num"].value_counts(dropna=False).to_frame("count"))

In [ ]:
df["attnets_num"].apply(type).value_counts()

In [ ]:
print("consensus_visits dtypes (before):")
print(df_consensus_visits.dtypes)

print("\nexecution_visits dtypes (before):")
print(df_execution_visits.dtypes)

print("consensus_visits:", df_consensus_visits.shape)
print("execution_visits:", df_execution_visits.shape)